In [1]:
import sys, time, itertools
import joblib
import pandas as pd
import numpy as np
import sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder, TargetEncoder,
                                   FunctionTransformer)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

random_state = 42
best_x = 7
data_dir = "/Users/shouryakulkarni/IDX-Exchange-Data-Science"
pd.set_option('display.max_columns', 50)

print("python", sys.version.split()[0], "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

python 3.12.12 | pandas 3.0.0 | scikit-learn 1.9.1


In [2]:
clean = pd.read_csv(f"{data_dir}/data/cleaned_sfr_districts.csv",
                    dtype={"PostalCode": str, "ElemDistrict": str, "HighDistrict": str},
                    parse_dates=["CloseDate"])
print(clean.shape)

(107416, 26)


In [3]:
target = "ClosePrice"

numeric_features = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSqFt",
                    "YearBuilt", "Latitude", "Longitude", "GarageSpaces", "Stories"]
skewed_features = ["LivingArea", "LotSqFt"]
other_numeric = [c for c in numeric_features if c not in skewed_features]
binary_features = ["PoolPrivateYN", "ViewYN", "FireplaceYN", "NewConstructionYN"]
low_card_categorical = ["CountyOrParish"]
high_card_categorical = ["City", "PostalCode", "ElemDistrict", "HighDistrict"]

features = numeric_features + binary_features + low_card_categorical + high_card_categorical

In [4]:
def fit_outlier_bounds(train, lo=0.005, hi=0.995):
    ppsf = train[target] / train["LivingArea"]
    return {"price_lo": train[target].quantile(lo), "price_hi": train[target].quantile(hi),
            "ppsf_lo": ppsf.quantile(lo), "ppsf_hi": ppsf.quantile(hi)}

def apply_outlier_bounds(data, b):
    ppsf = data[target] / data["LivingArea"]
    keep = (data[target].between(b["price_lo"], b["price_hi"]) &
            ppsf.between(b["ppsf_lo"], b["ppsf_hi"]))
    return data[keep]

def make_preprocessor():
    skewed = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ])
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    low_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False)),
    ])
    high_card = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target", TargetEncoder(target_type="continuous", random_state=random_state)),
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("skew", skewed, skewed_features),
        ("num", numeric, other_numeric),
        ("bin", "passthrough", binary_features),
        ("low", low_card, low_card_categorical),
        ("high", high_card, high_card_categorical),
    ])

def evaluate(y_true, y_pred):
    pct_err = np.abs(y_pred - y_true) / y_true
    return {"R2": r2_score(y_true, y_pred),
            "MAPE %": pct_err.mean() * 100,
            "MdAPE %": np.median(pct_err) * 100,
            "MAE $": mean_absolute_error(y_true, y_pred),
            "Within 20 %": (pct_err <= 0.20).mean() * 100}

def fit_and_score(estimator, train, eval_set):
    b = fit_outlier_bounds(train)
    tr, ev = apply_outlier_bounds(train, b), apply_outlier_bounds(eval_set, b)
    model = Pipeline([("prep", make_preprocessor()), ("model", clone(estimator))])
    model.fit(tr[features], np.log(tr[target]))
    pred = np.exp(model.predict(ev[features]))
    return evaluate(ev[target].values, pred), model, ev, pred

In [5]:
months = sorted(clean["CloseMonth"].unique())
tune_train = clean[clean["CloseMonth"].isin(months[-2 - best_x : -2])]
val = clean[clean["CloseMonth"] == months[-2]]
final_train = clean[clean["CloseMonth"].isin(months[-1 - best_x : -1])]
test = clean[clean["CloseMonth"] == months[-1]]
print("Validate on", months[-2], "| Test on", months[-1])

Validate on 2026-03 | Test on 2026-04


In [6]:
def make_grid(**options):
    keys = list(options)
    return [dict(zip(keys, values)) for values in itertools.product(*options.values())]

def tune(name, make_model, grid):
    rows = []
    for params in grid:
        start = time.time()
        m, _, _, _ = fit_and_score(make_model(**params), tune_train, val)
        rows.append({**params, "seconds": round(time.time() - start, 1), **m})
        print(f"{name} {params}: MdAPE {m['MdAPE %']:.2f}%")
    res = pd.DataFrame(rows)
    best = grid[res["MdAPE %"].idxmin()]
    print(f"\nBest {name} settings: {best}")
    return res.sort_values("MdAPE %").round(3), best

def make_lgbm(**p):
    return LGBMRegressor(**p, num_leaves=63, subsample=0.8, subsample_freq=1,
                         colsample_bytree=0.8, random_state=random_state,
                         n_jobs=-1, verbose=-1)

def make_xgb(**p):
    return XGBRegressor(**p, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                        random_state=random_state, n_jobs=-1)

def make_hgb(**p):
    return HistGradientBoostingRegressor(**p, early_stopping=False,
                                         random_state=random_state)

In [7]:
lgbm_results, lgbm_params = tune("LightGBM", make_lgbm, make_grid(
    max_depth=[6, 10, -1], learning_rate=[0.03, 0.1], n_estimators=[500, 1500]))
lgbm_results.head()

/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


LightGBM {'max_depth': 6, 'learning_rate': 0.03, 'n_estimators': 500}: MdAPE 8.73%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


LightGBM {'max_depth': 6, 'learning_rate': 0.03, 'n_estimators': 1500}: MdAPE 8.02%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


LightGBM {'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 500}: MdAPE 8.08%


... (15 outputs left out)

LightGBM {'max_depth': -1, 'learning_rate': 0.1, 'n_estimators': 500}: MdAPE 7.96%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


LightGBM {'max_depth': -1, 'learning_rate': 0.1, 'n_estimators': 1500}: MdAPE 7.81%

Best LightGBM settings: {'max_depth': -1, 'learning_rate': 0.1, 'n_estimators': 1500}


,max_depth,learning_rate,n_estimators,seconds,R2,MAPE %,MdAPE %,MAE $,Within 20 %
11,-1,0.10,1500,17.2,0.905,11.090,7.809,147835.257,85.362
5,10,0.03,1500,17.7,0.906,11.036,7.843,147390.106,85.256
3,6,0.10,1500,9.7,0.906,11.064,7.854,147691.520,85.221
7,10,0.10,1500,16.2,0.907,11.069,7.867,147011.356,85.476
9,-1,0.03,1500,17.9,0.905,11.047,7.895,147658.810,85.432


In [8]:
xgb_results, xgb_params = tune("XGBoost", make_xgb, make_grid(
    max_depth=[6, 10], learning_rate=[0.03, 0.1], n_estimators=[500, 1500]))
xgb_results.head()

/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


XGBoost {'max_depth': 6, 'learning_rate': 0.03, 'n_estimators': 500}: MdAPE 8.68%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


XGBoost {'max_depth': 6, 'learning_rate': 0.03, 'n_estimators': 1500}: MdAPE 8.04%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


XGBoost {'max_depth': 6, 'learning_rate': 0.1, 'n_estimators': 500}: MdAPE 7.87%


... (7 outputs left out)

XGBoost {'max_depth': 10, 'learning_rate': 0.1, 'n_estimators': 500}: MdAPE 7.80%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


XGBoost {'max_depth': 10, 'learning_rate': 0.1, 'n_estimators': 1500}: MdAPE 7.90%

Best XGBoost settings: {'max_depth': 10, 'learning_rate': 0.03, 'n_estimators': 1500}


,max_depth,learning_rate,n_estimators,seconds,R2,MAPE %,MdAPE %,MAE $,Within 20 %
5,10,0.03,1500,15.4,0.902,10.905,7.603,146801.949,85.556
3,6,0.10,1500,6.5,0.905,10.970,7.696,146892.775,85.688
4,10,0.03,500,5.4,0.899,11.086,7.781,149444.361,85.168
6,10,0.10,500,5.4,0.898,11.102,7.797,149663.753,85.194
2,6,0.10,500,2.4,0.902,11.262,7.865,150449.325,84.815


In [9]:
hgb_results, hgb_params = tune("HistGradientBoosting", make_hgb, make_grid(
    max_depth=[6, 10, None], learning_rate=[0.03, 0.1], max_iter=[500, 1500]))
hgb_results.head()

/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


HistGradientBoosting {'max_depth': 6, 'learning_rate': 0.03, 'max_iter': 500}: MdAPE 8.88%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


HistGradientBoosting {'max_depth': 6, 'learning_rate': 0.03, 'max_iter': 1500}: MdAPE 8.17%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


HistGradientBoosting {'max_depth': 6, 'learning_rate': 0.1, 'max_iter': 500}: MdAPE 8.16%


... (15 outputs left out)

HistGradientBoosting {'max_depth': None, 'learning_rate': 0.1, 'max_iter': 500}: MdAPE 8.05%


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


HistGradientBoosting {'max_depth': None, 'learning_rate': 0.1, 'max_iter': 1500}: MdAPE 7.83%

Best HistGradientBoosting settings: {'max_depth': 6, 'learning_rate': 0.1, 'max_iter': 1500}


,max_depth,learning_rate,max_iter,seconds,R2,MAPE %,MdAPE %,MAE $,Within 20 %
3,6.0,0.1,1500,33.3,0.904,11.109,7.819,148775.565,85.124
11,NaN,0.1,1500,39.3,0.906,11.053,7.826,147528.518,85.423
7,10.0,0.1,1500,37.0,0.905,11.080,7.892,148419.018,85.397
10,NaN,0.1,500,14.3,0.899,11.387,8.053,152573.889,84.287
2,6.0,0.1,500,11.9,0.897,11.480,8.162,153972.376,84.137


In [10]:
final_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                           max_features=0.6, n_jobs=-1,
                                           random_state=random_state),
    "LightGBM": make_lgbm(**lgbm_params),
    "XGBoost": make_xgb(**xgb_params),
    "HistGradientBoosting": make_hgb(**hgb_params),
}

val_rows, test_rows, fitted, test_preds = {}, {}, {}, {}
for name, est in final_models.items():
    start = time.time()
    val_rows[name], _, _, _ = fit_and_score(est, tune_train, val)
    test_rows[name], fitted[name], test_f, test_preds[name] = fit_and_score(est, final_train, test)
    print(f"done: {name} ({time.time() - start:.0f}s)")

comparison = pd.concat({
    "validation (Mar)": pd.DataFrame(val_rows).T,
    "test (Apr)": pd.DataFrame(test_rows).T,
}).round(3)
comparison

/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


done: Linear Regression (1s)


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


done: Random Forest (31s)


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


done: LightGBM (37s)


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


done: XGBoost (32s)


/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


done: HistGradientBoosting (68s)


R2  MAPE %  MdAPE %       MAE $  \
validation (Mar) Linear Regression     0.802  14.885   11.289  201318.702   
                 Random Forest         0.881  11.783    8.106  160612.927   
                 LightGBM              0.905  11.090    7.809  147835.257   
                 XGBoost               0.902  10.905    7.603  146801.949   
                 HistGradientBoosting  0.904  11.109    7.819  148775.565   
test (Apr)       Linear Regression     0.816  15.184   11.379  209325.074   
                 Random Forest         0.873  12.074    8.249  169032.049   
                 LightGBM              0.896  11.288    7.808  156060.660   
                 XGBoost               0.892  11.194    7.706  155926.164   
                 HistGradientBoosting  0.891  11.408    7.996  158362.434   

                                       Within 20 %  
validation (Mar) Linear Regression          75.183  
                 Random Forest              83.053  
                 LightGBM                   85.362  
                 XGBoost                    85.556  
                 HistGradientBoosting       85.124  
test (Apr)       Linear Regression          74.392  
                 Random Forest              82.890  
                 LightGBM                   85.174  
                 XGBoost                    85.296  
                 HistGradientBoosting       84.856

In [11]:
b = fit_outlier_bounds(final_train)
train_f = apply_outlier_bounds(final_train, b)
fit_rows = {}
for name, model in fitted.items():
    train_pred = np.exp(model.predict(train_f[features]))
    fit_rows[name] = {"train MdAPE %": evaluate(train_f[target].values, train_pred)["MdAPE %"],
                      "test MdAPE %": test_rows[name]["MdAPE %"]}
overfit = pd.DataFrame(fit_rows).T
overfit["gap"] = overfit["test MdAPE %"] - overfit["train MdAPE %"]
overfit.round(2)

,train MdAPE %,test MdAPE %,gap
Linear Regression,11.04,11.38,0.34
Random Forest,4.54,8.25,3.70
LightGBM,5.31,7.81,2.49
XGBoost,4.12,7.71,3.59
HistGradientBoosting,6.45,8.00,1.54


In [12]:
bands = pd.qcut(test_f[target], 5, labels=["Q1 cheapest", "Q2", "Q3", "Q4", "Q5 priciest"])
band_rows = []
for name, pred in test_preds.items():
    pct = (pred - test_f[target].values) / test_f[target].values * 100
    tmp = pd.DataFrame({"band": bands.values, "abs": np.abs(pct), "signed": pct})
    g = tmp.groupby("band", observed=True).agg(MdAPE=("abs", "median"), bias=("signed", "median"))
    g["model"] = name
    band_rows.append(g.reset_index())

band_table = pd.concat(band_rows).pivot(index="band", columns="model", values=["MdAPE", "bias"])
band_table.round(1)

MdAPE                                           \
model       HistGradientBoosting LightGBM Linear Regression Random Forest   
band                                                                        
Q1 cheapest                  7.0      7.0              10.6           7.2   
Q2                           6.3      5.9               9.7           6.2   
Q3                           7.0      7.0               9.9           7.1   
Q4                           8.9      8.8              11.8           9.1   
Q5 priciest                 12.4     12.4              16.5          13.3   

                                    bias                             \
model       XGBoost HistGradientBoosting LightGBM Linear Regression   
band                                                                  
Q1 cheapest     6.9                  0.6      0.7               2.8   
Q2              6.0                 -1.3     -1.0               0.6   
Q3              6.8                 -2.1     -2.3               0.7   
Q4              8.5                 -4.3     -3.9              -4.2   
Q5 priciest    11.9                 -7.8     -7.2             -12.4   

                                   
model       Random Forest XGBoost  
band                               
Q1 cheapest           1.6     0.8  
Q2                   -0.7    -1.0  
Q3                   -1.5    -2.2  
Q4                   -4.4    -3.8  
Q5 priciest          -9.3    -7.9

In [13]:
def backtest(estimator, data, X, n_cutoffs=3):
    months = sorted(data["CloseMonth"].unique())
    rows = []
    for tm in months[::-1][:n_cutoffs]:
        t = months.index(tm)
        tr = data[data["CloseMonth"].isin(months[t - X:t])]
        te = data[data["CloseMonth"] == tm]
        m, _, _, _ = fit_and_score(estimator, tr, te)
        rows.append({"test month": tm, **m})
    return pd.DataFrame(rows)

bt = pd.concat({name: backtest(final_models[name], clean, best_x)
                for name in ["Random Forest", "XGBoost", "LightGBM"]})

bt_mdape = bt.reset_index(level=0).pivot(index="test month", columns="level_0", values="MdAPE %")
bt_mdape.round(2)

/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/shouryakulkarni/miniforge3/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generato

level_0,LightGBM,Random Forest,XGBoost
test month,,,
2026-02,8.00,8.33,7.91
2026-03,7.81,8.11,7.60
2026-04,7.81,8.25,7.71


In [14]:
xgb_model = fitted["XGBoost"]
names = xgb_model.named_steps["prep"].get_feature_names_out()
importance = pd.Series(xgb_model.named_steps["model"].feature_importances_, index=names)
importance.sort_values(ascending=False).head(15).round(3)

high__PostalCode                          0.299
high__City                                0.118
skew__LivingArea                          0.037
low__CountyOrParish_Imperial              0.030
num__BathroomsTotalInteger                0.029
low__CountyOrParish_infrequent_sklearn    0.024
high__ElemDistrict                        0.023
high__HighDistrict                        0.020
low__CountyOrParish_Tulare                0.019
low__CountyOrParish_Santa Clara           0.018
low__CountyOrParish_Riverside             0.017
low__CountyOrParish_Marin                 0.015
low__CountyOrParish_Los Angeles           0.014
low__CountyOrParish_Fresno                0.014
low__CountyOrParish_San Mateo             0.013
dtype: float32

In [15]:
best_name = "XGBoost"   # chosen by validation MdAPE; revisit if the backtest says otherwise
joblib.dump({
    "model": fitted[best_name],
    "bounds": fit_outlier_bounds(final_train),
    "features": features,
    "train_months": months[-1 - best_x : -1],
    "test_month": months[-1],
}, f"{data_dir}/best_model.joblib")

comparison.reset_index().rename(columns={"level_0": "split", "level_1": "model"}) \
          .to_csv(f"{data_dir}/advanced_model_results.csv", index=False)
print("Saved best_model.joblib and advanced_model_results.csv")

Saved best_model.joblib and advanced_model_results.csv


## Week 7 Findings: Gradient Boosting

**Setup:** "+ school districts" features from week 6, same preprocessing and outlier rules. Each model tuned on March (depth, learning rate, number of trees), then retrained on Sep–Mar and tested once on April.
- LightGBM: max_depth = -1, learning_rate = 0.1, n_estimators = 1500
- XGBoost: max_depth = 10, learning_rate = 0.03, n_estimators = 1500
- HistGradientBoosting: max_depth = 6, learning_rate = 0.1, n_estimators = 1500

**Test results (April 2026)**
| Model | R² | MdAPE | Within 20% |
|---|---|---|---|
| Linear Regression | 0.816 | 11.38% | 74.4% |
| Random Forest | 0.873 | 8.25% | 82.9% |
| **XGBoost** | 0.892 | **7.71%** | **85.3%** |
| LightGBM | 0.896 | 7.81% | 85.2% |
| HistGradientBoosting | 0.891 | 8.00% | 84.9% |

**Best model: XGBoost.** Lowest validation MdAPE (7.60%), and it beat both the random forest (by 0.4–0.5 pts) and LightGBM (by 0.1–0.2 pts) in all 3 backtest months (Feb–Apr). Overall, typical error fell from 11.4% (week 4 baseline) to 7.7%, about a third less.

**Why boosting helps:** each new tree focuses on the errors of the previous ones, so hard cases such as expensive homes get more attention. Q5 (priciest 20%) error fell from 13.3% (RF) to 11.9%, and the underestimate shrank from −9.3% to −7.9%.

**Limitations:** still underestimates luxury homes by ~8%. XGBoost fits training data closely (train 4.1% vs. test 7.7%), but settings were chosen on unseen validation data and test performance is the best of any model. scikit-learn was upgraded to 1.9 during this notebook (earlier notebooks used 1.8).

**Saved:** best_model.joblib (full pipeline + outlier cutoffs + feature list) for the week 9 app.